In [ ]:
!pip install transformers
!pip install datasets
!pip install pandas
!pip install sklearn
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 whic

In [ ]:
import torch
import numpy as np
import pandas as pd
import os
import random

from transformers import T5ForConditionalGeneration, Trainer, TrainingArguments, T5Tokenizer
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import Dataset, DataLoader
from typing import List
from evaluate import load
from google.colab import drive
from datasets import Dataset, ClassLabel

In [ ]:
os.environ["WANDB_DISABLED"] = "true"
torch.cuda.empty_cache()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [ ]:
MODEL_NAME_QG = 'danielfdev/flan-t5-base-educational-question-generate'
MODEL_NAME_QA = 'danielfdev/flan-t5-base-educational-question-answer'
MODEL_NAME_DG = 'danielfdev/flan-t5-base-educational-distractor-generate'

MAX_LENGTH_INPUT = 512
MAX_LENGTH_OUTPUT = 256

In [ ]:
df = pd.read_csv('./sample_data/base_questoes.csv')
df.head()

,titulo_do_texto,texto,descritor
0,Dois amigos e um chato,Os dois estavam tomando um cafezinho no boteco...,D23 - Identificar os níveis de linguagem e/ou ...
1,Do que é feito o iogurte?,O iogurte é um alimento amplamente consumido p...,D23 - Identificar os níveis de linguagem e/ou ...
2,Feliz por não ser ninguém,[...] Acabo de voltar de uma viagem rumo ao co...,D23 - Identificar os níveis de linguagem e/ou ...
3,Anestesia,"O dentista, preocupado pela rejeição quase tot...",D22 - Reconhecer efeitos de humor e ironia.
4,Penso e passo,Quando penso que uma palavra pode mudar tudo n...,D21 - Reconhecer o efeito decorrente do empreg...


In [ ]:
tokenizer_qg = T5Tokenizer.from_pretrained(MODEL_NAME_QG)
tokenizer_qa = T5Tokenizer.from_pretrained(MODEL_NAME_QA)
tokenizer_dg = T5Tokenizer.from_pretrained(MODEL_NAME_DG)

model_qg = T5ForConditionalGeneration.from_pretrained(MODEL_NAME_QG).to(device)
model_qa = T5ForConditionalGeneration.from_pretrained(MODEL_NAME_QA).to(device)
model_dg = T5ForConditionalGeneration.from_pretrained(MODEL_NAME_DG).to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [ ]:
def generate_text(model, tokenizer, input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH_INPUT).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=3,
            no_repeat_ngram_size=3,
            early_stopping=True,
            max_length=MAX_LENGTH_OUTPUT
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def generate_distractors(input_text, correct_answer, answer_item, max_attempts=3):
    unique_distractors = set()

    for _ in range(max_attempts):
        inputs = tokenizer_dg(input_text, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH_INPUT).to(device)

        with torch.no_grad():
            outputs = model_dg.generate(
                **inputs,
                num_beams=5,
                no_repeat_ngram_size=3,
                early_stopping=True,
                max_length=MAX_LENGTH_OUTPUT,
                num_return_sequences=3
            )

        for output in outputs:
            decoded_text = tokenizer_dg.decode(output, skip_special_tokens=True).strip()
            distractor_list = [d.strip() for d in decoded_text.split(";") if d.strip()]
            unique_distractors.update(distractor_list)

        if len(unique_distractors) >= 3:
            break

    distractors_list = list(unique_distractors)[:3]

    item_to_index = {"a": 0, "b": 1, "c": 2, "d": 3}
    index = item_to_index[answer_item]

    distractors_list.insert(index, correct_answer)

    return str(distractors_list)

In [ ]:
def process_dataset(df):
    comandos = []
    respostas = []
    resposta_itens = []
    opcoes = []

    for _, row in df.iterrows():
        texto = row["texto"]
        descritor = row["descritor"]

        comando_input = f"Write a question about the following article: {descritor}, {texto}\n\nQuestion about the article:"
        comando = generate_text(model_qg, tokenizer_qg, comando_input)
        comandos.append(comando)

        # Gerar resposta com model_qa
        resposta_input = f"{comando}\n\nAnswer the question above based on this article: {texto}"
        resposta = generate_text(model_qa, tokenizer_qa, resposta_input)
        respostas.append(resposta)

        # Gerar uma letra aleatória para resposta_item
        resposta_item = random.choice(["a", "b", "c", "d"])
        resposta_itens.append(resposta_item)

        destratores_input = f"What would be incorrect answers to the question?\n\n{texto}\n\nQuestion: {comando}\n\correct answer: {resposta}"
        distratores = generate_distractors(destratores_input, resposta, resposta_item)
        opcoes.append(distratores)

    # Adicionando ao DataFrame
    df["comando"] = comandos
    df["resposta"] = respostas
    df["resposta_item"] = resposta_itens
    df["opcoes"] = opcoes

    return df

In [ ]:
new_df = process_dataset(df)

In [ ]:
new_df.to_csv("dataset_gerado.csv", index=False, encoding="utf-8")